# 01 – Detector Walkthrough

> **Research prototype notebook.**  This notebook shows how to import and run the
> THUNBIT detectors on simple synthetic demand series.  Outputs are illustrative;
> they will vary across Python/package versions and are not validated against
> real-world data.

## What this notebook covers

1. Importing the package
2. Generating a simple synthetic demand series
3. Running all four detector variants
4. Inspecting the output DataFrame
5. A simple confidence plot (requires `matplotlib`)

## Setup

```bash
pip install -e .            # from the repository root
# optional, for the plot cell:
pip install matplotlib
```

In [ ]:
import numpy as np
import pandas as pd

from thunbit import (
    DemandStateDetector,
    StabilizedDemandDetector,
    StabilizedDemandDetectorV41,
    StabilizedDemandDetectorV42,
    StabilizedDemandDetectorV44,
)

print('thunbit imported successfully')

## 1. Synthetic demand series

We generate two 400-day demand series:
- **Stable series** – stationary normal demand, no injected break.
- **Mean-shift series** – demand mean increases ~60% at day 200.

The detectors need `window_long + window_short = 111` observations before
producing the first output row, so 400 days gives ~289 output rows.

In [ ]:
rng = np.random.default_rng(42)
N = 400
BREAK_DAY = 200

dates = pd.date_range('2023-01-01', periods=N, freq='D')

# Stable: constant mean / variance, no injected break
stable_series = rng.normal(loc=100.0, scale=10.0, size=N).clip(0)

# Mean-shift: demand mean jumps ~60% at day 200
pre_shift  = rng.normal(loc=100.0, scale=10.0, size=BREAK_DAY).clip(0)
post_shift = rng.normal(loc=160.0, scale=12.0, size=N - BREAK_DAY).clip(0)
shift_series = np.concatenate([pre_shift, post_shift])

print(f'Stable series  : mean={stable_series.mean():.1f}  std={stable_series.std():.1f}')
print(f'Shift series   : pre-break mean={shift_series[:BREAK_DAY].mean():.1f}',
      f' post-break mean={shift_series[BREAK_DAY:].mean():.1f}')

## 2. Baseline detector (`DemandStateDetector`)

The baseline uses direct threshold comparison — no state machine, no hysteresis.
It is the fastest to respond but produces many fragmented false-alert clusters on
stable series.

In [ ]:
det_base = DemandStateDetector()

# Use detect_rolling for the baseline (no state-machine variant)
df_base_stable = det_base.detect_rolling(stable_series, dates=dates)
df_base_shift  = det_base.detect_rolling(shift_series,  dates=dates)

print('--- Baseline: stable series ---')
print(df_base_stable['state'].value_counts().to_string())
print(f"Alert days: {(df_base_stable['state'] != 'STABLE').sum()} / {len(df_base_stable)}")

print()
print('--- Baseline: mean-shift series ---')
print(df_base_shift['state'].value_counts().to_string())

## 3. V4 stabilized detector (`StabilizedDemandDetector`)

V4 adds hysteresis, smoothing, and confirmation to reduce noisy transitions.
It roughly halves false-alert clusters on stable series but introduces detection
delay on gradual-drift and intermittent scenarios.

In [ ]:
det_v4 = StabilizedDemandDetector()

df_v4_stable = det_v4.detect_rolling_stabilized(stable_series, dates=dates)
df_v4_shift  = det_v4.detect_rolling_stabilized(shift_series,  dates=dates)

print('--- V4: stable series ---')
print(df_v4_stable['state'].value_counts().to_string())

print()
print('--- V4: mean-shift series ---')
print(df_v4_shift['state'].value_counts().to_string())

## 4. V4.1 detector (`StabilizedDemandDetectorV41`)

V4.1 adds a post-return cooldown and relaxes confirmation thresholds, recovering
some detection speed lost in V4.  Stable-series alert burden remains largely
unchanged from the baseline — this motivated the score-normalization approach
introduced in V4.2.

In [ ]:
det_v41 = StabilizedDemandDetectorV41()

df_v41_stable = det_v41.detect_rolling_stabilized(stable_series, dates=dates)
df_v41_shift  = det_v41.detect_rolling_stabilized(shift_series,  dates=dates)

print('--- V4.1: stable series ---')
print(df_v41_stable['state'].value_counts().to_string())

print()
print('--- V4.1: mean-shift series ---')
print(df_v41_shift['state'].value_counts().to_string())

## 5. V4.2 detector (`StabilizedDemandDetectorV42`)

V4.2 introduced baseline-normalized scoring: raw confidence is compared against
a rolling median of the SKU's own recent confidence.  This yielded a major
reduction in stable-series false alerts but over-damped real-break detection.
V4.2 is kept as a historical reference.

The output DataFrame includes additional columns:
- `baseline_confidence` — rolling median of recent raw confidence
- `normalized_confidence` — excess above baseline, scaled to [0, 1]
- `confidence` — smoothed normalized confidence used by the state machine

In [ ]:
det_v42 = StabilizedDemandDetectorV42()

df_v42_stable = det_v42.detect_rolling_stabilized(stable_series, dates=dates)
df_v42_shift  = det_v42.detect_rolling_stabilized(shift_series,  dates=dates)

print('--- V4.2: stable series ---')
print(df_v42_stable['state'].value_counts().to_string())
print(f"Alert days: {(df_v42_stable['state'] != 'STABLE').sum()} / {len(df_v42_stable)}")

print()
print('--- V4.2: mean-shift series (first alert after break) ---')
post_break = df_v42_shift[df_v42_shift['t'] >= BREAK_DAY]
alerts = post_break[post_break['state'].isin(['DRIFT', 'SHIFT'])]
if len(alerts):
    first = alerts.iloc[0]
    print(f"  First alert: day {first['t']}  (delay {int(first['t']) - BREAK_DAY} days)  state={first['state']}")
else:
    print('  Break not detected.')

print()
print('--- V4.2 extra columns (last 3 rows) ---')
cols = ['t', 'state', 'raw_confidence', 'baseline_confidence', 'normalized_confidence', 'confidence']
print(df_v42_shift[cols].tail(3).to_string(index=False))

## 6. V4.4 detector (`StabilizedDemandDetectorV44`) — current recommended experimental variant

V4.4 (the V4.4b calibration) keeps V4.3's normalized-score setup and tightens state-entry calibration by:
- using the **25th-percentile** rolling baseline instead of the median,
  so genuine breaks produce a larger relative excess signal;
- adding **warmup suppression** for the first 28 output rows;
- extending the baseline window (28 days vs 21) and cooldown (7 vs 5).

V4.4 is the **current recommended experimental operating point**.  It is not
production-ready and stable-series false alerts remain an open problem.

In [ ]:
det_v44 = StabilizedDemandDetectorV44()

df_v44_stable = det_v44.detect_rolling_stabilized(stable_series, dates=dates)
df_v44_shift  = det_v44.detect_rolling_stabilized(shift_series,  dates=dates)

print('--- V4.4: stable series ---')
print(df_v44_stable['state'].value_counts().to_string())
print(f"Alert days: {(df_v44_stable['state'] != 'STABLE').sum()} / {len(df_v44_stable)}")

print()
print('--- V4.4: mean-shift series ---')
print(df_v44_shift['state'].value_counts().to_string())
post_break = df_v44_shift[df_v44_shift['t'] >= BREAK_DAY]
alerts = post_break[post_break['state'].isin(['DRIFT', 'SHIFT'])]
if len(alerts):
    first = alerts.iloc[0]
    print(f"  First alert: day {first['t']}  (delay {int(first['t']) - BREAK_DAY} days)  state={first['state']}")
else:
    print('  Break not detected.')

## 7. Inspect the output DataFrame

All detectors return a `pd.DataFrame` with at least these columns:

| Column | Description |
|--------|-------------|
| `t` | Day index in the original series |
| `date` | Date label (if provided) |
| `state` | `STABLE` / `DRIFT` / `SHIFT` |
| `raw_confidence` | Raw composite score (0–1) |
| `confidence` | Smoothed score used by state machine |
| `pss` | Probable stable stock level (rough heuristic) |
| `horizon` | Recommended planning horizon for current state |
| `action` | Recommended action string for current state |
| `evidence` | Dict of individual channel scores |

V4.2, V4.3, and V4.4 add: `baseline_confidence`, `normalized_confidence`, `prev_state`.

In [ ]:
display_cols = [
    't', 'date', 'state',
    'raw_confidence', 'baseline_confidence', 'normalized_confidence', 'confidence',
    'pss', 'horizon', 'action',
]

print('Last 8 rows of V4.4 output (mean-shift series):')
print(df_v44_shift[display_cols].tail(8).to_string(index=False))

## 8. Side-by-side stable-series summary

How many days does each detector spend in a non-STABLE state on the
stable (no-break) series?  Lower is better.

In [ ]:
results = {
    'Baseline':  df_base_stable,
    'V4':        df_v4_stable,
    'V4.1':      df_v41_stable,
    'V4.2':      df_v42_stable,
    'V4.4':      df_v44_stable,
}

rows = []
for name, df in results.items():
    n_total = len(df)
    n_alert = (df['state'] != 'STABLE').sum()
    rows.append({'Detector': name, 'Total rows': n_total,
                 'Alert days': n_alert, 'Alert %': f"{n_alert/n_total*100:.1f}%"})

summary = pd.DataFrame(rows).set_index('Detector')
print('Stable-series false-alert summary (single seed, illustrative):')
print(summary.to_string())
print()
print('Note: V4.2/V4.4 show the largest reductions in alert burden.')
print('V4.4 (V4.4b calibration) is the recommended experimental operating point.')

## 9. Confidence plot (requires `matplotlib`)

The cell below plots `confidence` over time for V4.4 on the mean-shift series.
Install `matplotlib` to run it:

```bash
pip install matplotlib
```

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    # Top panel: demand series
    ax = axes[0]
    ax.plot(range(N), shift_series, lw=0.8, color='steelblue', label='Demand')
    ax.axvline(BREAK_DAY, color='red', lw=1.5, ls='--', label=f'Break (day {BREAK_DAY})')
    ax.set_ylabel('Demand')
    ax.set_title('Mean-shift series with V4.4 detection')
    ax.legend(fontsize=9)

    # Bottom panel: confidence and state
    ax2 = axes[1]
    t = df_v44_shift['t'].values
    conf = df_v44_shift['confidence'].values
    raw  = df_v44_shift['raw_confidence'].values
    state = df_v44_shift['state'].values

    state_colors = {'STABLE': '#d4edda', 'DRIFT': '#fff3cd', 'SHIFT': '#f8d7da'}
    for i in range(len(t) - 1):
        ax2.axvspan(t[i], t[i+1], alpha=0.3, color=state_colors.get(state[i], 'white'), lw=0)

    ax2.plot(t, raw,  lw=0.8, color='grey',      alpha=0.7, label='raw_confidence')
    ax2.plot(t, conf, lw=1.5, color='darkorange', label='confidence (smoothed norm.)')
    ax2.axvline(BREAK_DAY, color='red', lw=1.5, ls='--')
    ax2.set_ylabel('Confidence score')
    ax2.set_xlabel('Day')
    ax2.set_ylim(-0.05, 1.05)

    patches = [mpatches.Patch(color=c, alpha=0.5, label=s) for s, c in state_colors.items()]
    ax2.legend(handles=patches + ax2.get_lines()[:2], fontsize=8, loc='upper left')

    plt.tight_layout()
    plt.savefig('v43_walkthrough.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Plot saved to v43_walkthrough.png')

except ImportError:
    print('matplotlib not installed – skipping plot.')
    print('Run:  pip install matplotlib')

---

## Summary

| Detector | Mechanism | Stable alert burden | Break speed |
|----------|-----------|:-------------------:|:-----------:|
| Baseline | Direct threshold | High | Fast |
| V4 | Hysteresis + smoothing | Moderate | Slower |
| V4.1 | + Cooldown, relaxed thresholds | Moderate | Moderate |
| V4.2 | Baseline-normalized (median) | Very low | Slow (over-damped) |
| **V4.4** | **V4.3 normalized score + stricter state calibration** | **Lower** | **Good** |

V4.4 is the current recommended experimental tradeoff.  See `docs/benchmarking.md`
for the full simulation results and `docs/limitations.md` for open problems.

**Next:** `02_benchmark_iterations.ipynb` – reproducible benchmark flow
across multiple synthetic scenarios.